PHASE 1: DATASET HARMONIZATION AND INGESTION STRATEGY
================================================================================
Thesis: 3D Resection & Surgical Planning of Brain Tumors Using DL + RL
Author: B.Tech Final Year | AI & Data Science
Framework: PyTorch + MONAI  |  Target: 16 GB T4 GPU (Kaggle/Colab)

1.1 — MATHEMATICAL FOUNDATION
-----------------------
1. Resampling to 2mm isotropic:
   Given voxel spacing (s_x, s_y, s_z), the resampled shape is:
   D' = round(D * s_z / 2),  H' = round(H * s_y / 2),  W' = round(W * s_x / 2)
   This ensures all volumes live in the same metric coordinate space.

2. Intensity Normalization (per-channel Z-score):
   x_norm = (x - μ) / σ  for each channel independently
   where μ = mean(non-zero voxels), σ = std(non-zero voxels)
   This is mandatory for multi-site harmonization: different scanners produce
   different absolute Hounsfield/signal-intensity distributions.

3. 2.5D Bounding-Box Extraction:
   Given a binary label mask L ∈ {0,1}^(D×H×W), compute the tight bounding
   box (z_min,z_max, y_min,y_max, x_min,x_max) of all non-zero voxels.
   The three central slices intersecting this box:
     - Axial:    I_ax = volume[:, :, z_c] where z_c = (z_min+z_max)//2
     - Coronal:  I_co = volume[:, y_c, :] where y_c = (y_min+y_max)//2
     - Sagittal: I_sa = volume[x_c, :, :] where x_c = (x_min+x_max)//2
   Stack → tensor of shape (3, H, W) → 3-channel "pseudo-RGB" analogous to
   ImageNet-pretrained 2D networks, but encoding orthogonal anatomical planes.
   Memory savings: 256³×4 float32 ≈ 256 MB → 3×256×256×4 bytes ≈ 0.75 MB

In [1]:
!pip install -q "monai[all]" gymnasium nibabel SimpleITK

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.4/54.4 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 266.5/266.5 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.9/80.9 MB 21.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 MB 27.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.0/28.0 MB 61.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 MB 32.6 MB/s eta 0:00:00:00:0100:01
   ━

In [2]:
import os
import json
import glob
import numpy as np
import torch
from pathlib import Path
from typing import Dict, List, Tuple, Optional

# ─── MONAI imports ────────────────────────────────────────────────────────────
import monai
from monai.data import CacheDataset, DataLoader
from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    Spacingd,
    Orientationd,
    NormalizeIntensityd,
    ScaleIntensityRangePercentilesd,
    CropForegroundd,
    RandFlipd,
    RandRotate90d,
    RandShiftIntensityd,
    ToTensord,
    EnsureTyped,
    MapTransform,
)
from monai.utils import set_determinism

# ─── Reproducibility seed ─────────────────────────────────────────────────────
set_determinism(seed=42)
torch.manual_seed(42)
np.random.seed(42)

<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
2026-04-11 17:50:08.766224: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775929808.954953      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775929809.003438      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775929809.443871      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775929809.443912      55 computation_placer.cc:1

In [3]:
CFG = {
    # ── Paths ──────────────────────────────────────────────────────────────────
    "ROOT_REMIND":  "/kaggle/input/datasets/thesukuna/remind-dataset-for-deep-learning/ReMIND2Reg/imagesTr",
    "ROOT_BRATS":   "/kaggle/input/datasets/awsaf49/brats20-dataset-training-validation/BraTS2020_TrainingData",  # Update on Kaggle
    "OUTPUT_DIR":   "/kaggle/working/pipeline_outputs",

    # ── Resampling ─────────────────────────────────────────────────────────────
    "VOXEL_SPACING": (2.0, 2.0, 2.0),  # 2 mm isotropic

    # ── 2.5D slice size after resize ───────────────────────────────────────────
    "SLICE_SIZE": 256,   # Final H×W of each extracted 2D slice

    # ── DataLoader ─────────────────────────────────────────────────────────────
    "BATCH_SIZE": 8,     # 8 × (3×256×256 float16) ≈ 192 MB — safe on T4
    "NUM_WORKERS": 2,
    "CACHE_RATE": 1.0,   # Cache all items post-transform (fits in RAM on Colab)

    # ── BraTS label mapping ────────────────────────────────────────────────────
    # BraTS 2021 labels: 0=BG, 1=NCR/NET (Necrotic Core), 2=ED (Edema), 4=ET
    # We remap 4→3 for a clean 0-3 integer label space
    "BRATS_LABEL_MAP": {0: 0, 1: 1, 2: 2, 4: 3},

    # ── ReMIND2Reg modality suffixes ────────────────────────────────────────────
    # _0000 = intraoperative 3D US (fixed, post-resection)
    # _0001 = pre-operative ceT1 MRI (moving)
    # _0002 = pre-operative T2 MRI   (moving, not always present)
    "REMIND_US_SUFFIX":  "_0000.nii",
    "REMIND_T1_SUFFIX":  "_0001.nii",
    "REMIND_T2_SUFFIX":  "_0002.nii",
}

os.makedirs(CFG["OUTPUT_DIR"], exist_ok=True)

## 1.2 — BUILD ReMIND2Reg DATA DICTIONARY

 ReMIND2Reg stores patient cases as:
 -  ReMIND2Reg_{PPPP}_{MMMM}.nii
 -  where PPPP = patient ID (0000–0102), MMMM = modality (0000/0001/0002)

 We group files by patient, then build (US, ceT1, T2) triplets.
 Some patients lack ceT1 or T2 — we fill with None and handle gracefully.

## 1.3 — BUILD BraTS 2021 DATA DICTIONARY
 BraTS 2021 folder structure (per patient):
  - BraTS2021_XXXXX/
   -  BraTS2021_XXXXX_t1.nii
   -  BraTS2021_XXXXX_t1ce.nii
   -  BraTS2021_XXXXX_t2.nii
   -  BraTS2021_XXXXX_flair.nii
   -  BraTS2021_XXXXX_seg.nii  <- label mask

In [4]:
def build_remind_data_list(root: str) -> List[Dict]:
    """
    Parse the ReMIND2Reg imagesTr directory and build a list of dicts,
    each representing one patient with keys: 'us', 'cet1', 't2'.

    For ReMIND2Reg the key challenge is that the pair we care about for
    the surgical planning pipeline is:
      - 'us'  : ground-truth post-resection state (the resection cavity IS
                 observable in the iUS image as a low-echo void)
      - 'cet1': pre-operative reference for tumor delineation
      - 't2'  : supplementary pre-operative contrast (used when available)

    Returns
    -------
    List of dicts with structure:
        {
          'image':  path_to_cet1_or_t2,   # primary MR input
          'us':     path_to_us,            # reference (not fed to network)
          'has_t2': bool,
          'patient_id': str
        }
    """
    img_dir = os.path.join(root, "imagesTr")
    # Collect all .nii files
    all_files = sorted(glob.glob(os.path.join(img_dir, "*.nii")))

    # Group by patient ID (first 4-digit field after "ReMIND2Reg_")
    from collections import defaultdict
    patients = defaultdict(dict)
    for fpath in all_files:
        basename = os.path.basename(fpath)
        # e.g. "ReMIND2Reg_0005_0001.nii"
        parts = basename.replace(".nii", "").split("_")
        # parts = ["ReMIND2Reg", "0005", "0001"]
        if len(parts) < 3:
            continue
        patient_id = parts[1]      # "0005"
        modality   = parts[2]      # "0000", "0001", "0002"
        patients[patient_id][modality] = fpath

    data_list = []
    for pid, modalities in patients.items():
        us   = modalities.get("0000", None)
        cet1 = modalities.get("0001", None)
        t2   = modalities.get("0002", None)

        # We need at least one MR modality for input
        primary_mr = cet1 if cet1 is not None else t2
        if primary_mr is None or us is None:
            continue   # Skip patients with no MR or no iUS

        data_list.append({
            "image":      primary_mr,
            "us":         us,
            "has_t2":     (t2 is not None),
            "patient_id": pid,
        })

    print(f"[ReMIND2Reg] Found {len(data_list)} usable patient cases.")
    return data_list


In [5]:
def build_brats_data_list(root: str, max_cases: int = 1000) -> List[Dict]:
    """
    Enumerate BraTS 2021 cases. Returns list of dicts with multi-modal
    image paths and segmentation label path.

    We treat the four modalities (T1, T1ce, T2, FLAIR) as 4 input channels.
    The label mask has classes 0,1,2,4 which we remap to 0,1,2,3.

    Parameters
    ----------
    root      : Root folder containing per-patient subdirectories.
    max_cases : Cap for quick debugging; set to a large number for full training.
    """
    cases = sorted([
        d for d in os.listdir(root)
        if os.path.isdir(os.path.join(root, d)) and "BraTS2021" in d
    ])[:max_cases]

    data_list = []
    for case_name in cases:
        case_dir = os.path.join(root, case_name)
        entry = {
            "t1":    os.path.join(case_dir, f"{case_name}_t1.nii"),
            "t1ce":  os.path.join(case_dir, f"{case_name}_t1ce.nii"),
            "t2":    os.path.join(case_dir, f"{case_name}_t2.nii"),
            "flair": os.path.join(case_dir, f"{case_name}_flair.nii"),
            "label": os.path.join(case_dir, f"{case_name}_seg.nii"),
        }
        # Validate all files exist
        if all(os.path.exists(v) for v in entry.values()):
            data_list.append(entry)

    print(f"[BraTS2021] Found {len(data_list)} valid cases.")
    return data_list

## 1.4 — CUSTOM 2.5D SLICE EXTRACTION TRANSFORM
 This is the core memory-optimization technique. Instead of processing the full
 256³ volume through the CNN (which would require ~4 GB VRAM per volume), we
 extract three orthogonal 2D slices centered on the tumor's bounding box.

 The resulting tensor has shape (3, H, W) — analogous to an RGB image but
 each channel encodes a different anatomical perspective:
-   Ch 0: Axial    (transverse plane)    — best for L/R symmetry analysis
-   Ch 1: Coronal  (frontal plane)       — best for superior/inferior extent
-   Ch 2: Sagittal (lateral plane)       — best for A/P surgical corridor

 This dramatically reduces memory while retaining global spatial context
 across all three anatomical axes simultaneously.

In [6]:
class Extract25DSlicesd(MapTransform):
    """
    Custom MONAI MapTransform to extract a 2.5D pseudo-RGB tensor from a
    3D volume by taking the central axial, coronal, and sagittal slices
    through the tumor's bounding box centroid.

    If no label is provided (inference mode), uses the volume centroid.

    Input  key 'image' shape: (C, D, H, W)  [C channels, typically 1 or 4]
    Output key 'image' shape: (3, SLICE_SIZE, SLICE_SIZE)

    The three slices are each resized to SLICE_SIZE × SLICE_SIZE using
    bilinear interpolation to ensure a uniform spatial resolution.
    """

    def __init__(self, keys, label_key="label", slice_size=256, allow_missing_keys=True):
        super().__init__(keys, allow_missing_keys=allow_missing_keys)
        self.label_key  = label_key
        self.slice_size = slice_size

    def _resize_slice(self, slc: np.ndarray) -> np.ndarray:
        """
        Resize a 2D numpy slice (H×W) to (self.slice_size × self.slice_size)
        via PIL bilinear interpolation. Preserves float32 precision.
        """
        from PIL import Image
        # Normalize to [0,255] for PIL, then rescale back to float32
        mn, mx = slc.min(), slc.max()
        if mx - mn > 1e-6:
            slc_uint8 = ((slc - mn) / (mx - mn) * 255).astype(np.uint8)
        else:
            slc_uint8 = np.zeros_like(slc, dtype=np.uint8)
        pil_img = Image.fromarray(slc_uint8).resize(
            (self.slice_size, self.slice_size), resample=Image.BILINEAR
        )
        slc_resized = np.array(pil_img, dtype=np.float32) / 255.0
        # Rescale back to original intensity range
        return slc_resized * (mx - mn) + mn

    def _get_bbox_centroid(self, label: np.ndarray) -> Tuple[int, int, int]:
        """
        Compute the centroid of the bounding box of all non-zero voxels
        in the label mask.

        Returns (z_c, y_c, x_c) indices into the (D, H, W) volume.
        If no tumor voxels found, returns the volume centroid.
        """
        # label shape: (D, H, W) after squeezing channel dim
        nz = np.argwhere(label > 0)
        if len(nz) == 0:
            # No tumor — use center of volume
            D, H, W = label.shape
            return D // 2, H // 2, W // 2

        z_min, y_min, x_min = nz.min(axis=0)
        z_max, y_max, x_max = nz.max(axis=0)

        z_c = int((z_min + z_max) // 2)
        y_c = int((y_min + y_max) // 2)
        x_c = int((x_min + x_max) // 2)
        return z_c, y_c, x_c


    def __call__(self, data: Dict) -> Dict:
        d = dict(data)

        for key in self.key_iterator(d):
            img = d[key]  # Shape: (C, D, H, W) — MONAI channel-first convention

            # ── Convert to numpy if still a tensor ────────────────────────────
            if isinstance(img, torch.Tensor):
                img_np = img.numpy()
            else:
                img_np = np.array(img)

            C, D, H, W = img_np.shape

            # ── Get bounding box centroid from label (if available) ────────────
            if self.label_key in d:
                lbl = d[self.label_key]
                if isinstance(lbl, torch.Tensor):
                    lbl_np = lbl.numpy()
                else:
                    lbl_np = np.array(lbl)
                # lbl shape: (1, D, H, W) → squeeze to (D, H, W)
                lbl_3d = np.squeeze(lbl_np)
                z_c, y_c, x_c = self._get_bbox_centroid(lbl_3d)
            else:
                z_c, y_c, x_c = D // 2, H // 2, W // 2

            # ── Clamp indices to valid range ───────────────────────────────────
            z_c = np.clip(z_c, 0, D - 1)
            y_c = np.clip(y_c, 0, H - 1)
            x_c = np.clip(x_c, 0, W - 1)

            # ── For multi-channel input, use first channel (or average) ─────────
            # For BraTS (C=4): we use the T1ce channel (index 1) as it best
            # highlights the enhancing tumor core for slice centroid extraction.
            ch_idx = min(1, C - 1)
            vol = img_np[ch_idx]  # (D, H, W)

            # ── Extract three orthogonal slices ────────────────────────────────
            # Axial slice: volume viewed from above → shape (H, W)
            axial_slc    = vol[z_c, :, :]   # ← fix the z plane, sweep H×W
            # Coronal slice: volume viewed from front → shape (D, W)
            coronal_slc  = vol[:, y_c, :]   # ← fix y plane, sweep D×W
            # Sagittal slice: volume viewed from side → shape (D, H)
            sagittal_slc = vol[:, :, x_c]   # ← fix x plane, sweep D×H

            # ── Resize each slice to (SLICE_SIZE × SLICE_SIZE) ────────────────
            axial_r    = self._resize_slice(axial_slc)
            coronal_r  = self._resize_slice(coronal_slc)
            sagittal_r = self._resize_slice(sagittal_slc)

            # ── Stack → shape (3, SLICE_SIZE, SLICE_SIZE) ─────────────────────
            pseudo_25d = np.stack([axial_r, coronal_r, sagittal_r], axis=0)  # (3, S, S)

            d[key] = torch.from_numpy(pseudo_25d.astype(np.float32))

        return d

## 1.5 — UNIFIED MONAI TRANSFORM PIPELINES

In [7]:
def get_brats_transforms(
    target_spacing: Tuple[float, float, float] = (2.0, 2.0, 2.0),
    slice_size: int = 256,
    is_train: bool = True,
) -> Compose:
    """
    BraTS 2021 transform pipeline.

    Input:  dict with keys 't1','t1ce','t2','flair','label' — paths to NIfTI files
    Output: dict with keys 'image' (3, S, S) and 'label' (3, S, S) as tensors

    Transforms:
      1. LoadImaged      — load all NIfTI files into numpy arrays
      2. EnsureChannelFirstd — add channel dim if missing
      3. Spacingd        — resample to 2mm isotropic (bilinear for images, NN for labels)
      4. Orientationd    — canonical RAS orientation
      5. NormalizeIntensityd — per-channel Z-score normalization (nonzero voxels)
      6. CropForegroundd — crop tight box around nonzero brain voxels
      7. [Train only] Augmentations: random flip, rotate90, intensity shift
      8. Extract25DSlicesd — extract 3-slice pseudo-RGB tensor (memory reduction)
    """
    # We'll concatenate all 4 modalities into a single 'image' key
    # using MONAI's ConcatItemsd after loading

    from monai.transforms import ConcatItemsd, DeleteItemsd

    # Step A: Build base transforms (common to train + val)
    base_transforms = [
        # 1. Load each modality as separate key
        LoadImaged(keys=["t1", "t1ce", "t2", "flair", "label"],
                   ensure_channel_first=True, image_only=False),
        # 2. Resample to 2mm isotropic
        #    pixdim for labels uses nearest-neighbor (mode="nearest") to
        #    preserve integer class values
        Spacingd(keys=["t1", "t1ce", "t2", "flair", "label"],
                 pixdim=target_spacing,
                 mode=["bilinear", "bilinear", "bilinear", "bilinear", "nearest"]),
        # 3. Canonical orientation (RAS = Right-Anterior-Superior)
        Orientationd(keys=["t1", "t1ce", "t2", "flair", "label"],
                     axcodes="RAS"),
        # 4. Per-modality Z-score normalization on nonzero voxels
        NormalizeIntensityd(keys=["t1", "t1ce", "t2", "flair"],
                            nonzero=True, channel_wise=True),
        # 5. Concatenate 4 modalities → single 'image' tensor of shape (4, D, H, W)
        ConcatItemsd(keys=["t1", "t1ce", "t2", "flair"], name="image", dim=0),
        # 6. Remove individual modality keys to save memory
        DeleteItemsd(keys=["t1", "t1ce", "t2", "flair"]),
        # 7. Crop to foreground (removes skull-stripped black background)
        CropForegroundd(keys=["image", "label"], source_key="image",
                        margin=5),
    ]

    # Step B: Training augmentations (randomized)
    augment_transforms = [
        RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=0),
        RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=1),
        RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=2),
        RandRotate90d(keys=["image", "label"], prob=0.5, max_k=3),
        RandShiftIntensityd(keys=["image"], offsets=0.1, prob=0.5),
    ] if is_train else []

    # Step C: 2.5D extraction + type conversion
    final_transforms = [
        # Extract 3 orthogonal slices → shape (3, slice_size, slice_size)
        Extract25DSlicesd(keys=["image"], label_key="label",
                          slice_size=slice_size),
        EnsureTyped(keys=["image", "label"], dtype=torch.float32),
    ]

    return Compose(base_transforms + augment_transforms + final_transforms)


def get_remind_transforms(
    target_spacing: Tuple[float, float, float] = (2.0, 2.0, 2.0),
    slice_size: int = 256,
    is_train: bool = True,
) -> Compose:
    """
    ReMIND2Reg transform pipeline.

    Input:  dict with keys 'image' (ceT1 or T2 path), 'us' (iUS path)
    Output: dict with keys:
      'image' → (3, S, S) tensor — the pre-operative MR in 2.5D
      'us'    → (3, S, S) tensor — the intra-operative iUS in 2.5D
                (serves as ground-truth resection reference)

    NOTE: Since ReMIND2Reg has no explicit segmentation masks, the centroid
    for 2.5D extraction is derived from the iUS image's non-zero region, which
    represents the post-resection cavity area.
    """
    from monai.transforms import ScaleIntensityRangePercentilesd

    base_transforms = [
        # Load both MR and iUS images
        LoadImaged(keys=["image", "us"], ensure_channel_first=True,
                   image_only=False),
        # Resample both to 2mm isotropic
        Spacingd(keys=["image", "us"], pixdim=target_spacing,
                 mode=["bilinear", "bilinear"]),
        # Canonical orientation
        Orientationd(keys=["image", "us"], axcodes="RAS"),
        # Normalize MR with Z-score
        NormalizeIntensityd(keys=["image"], nonzero=True, channel_wise=True),
        # Normalize US with percentile (US intensity is not Gaussian)
        ScaleIntensityRangePercentilesd(
            keys=["us"], lower=1, upper=99,
            b_min=0.0, b_max=1.0, clip=True
        ),
    ]

    augment_transforms = [
        RandFlipd(keys=["image", "us"], prob=0.5, spatial_axis=0),
        RandFlipd(keys=["image", "us"], prob=0.5, spatial_axis=1),
        RandShiftIntensityd(keys=["image"], offsets=0.15, prob=0.5),
    ] if is_train else []

    final_transforms = [
        # For ReMIND2Reg, use the US image's non-zero region to find the
        # surgical cavity centroid (where the resection happened)
        Extract25DSlicesd(keys=["image", "us"],
                          label_key="us",   # US non-zero = resection region
                          slice_size=slice_size),
        EnsureTyped(keys=["image", "us"], dtype=torch.float32),
    ]

    return Compose(base_transforms + augment_transforms + final_transforms)

## 1.6 — UNIFIED CACHEDATASET BUILDER

In [8]:
def build_brats_dataloaders(
    root: str,
    val_fraction: float = 0.1,
    max_cases: int = 1000,
) -> Tuple[DataLoader, DataLoader]:
    """
    Build CacheDatset-backed DataLoaders for BraTS 2021.

    CacheDataset pre-transforms all items once and caches them in RAM.
    This eliminates repeated IO + transform overhead at the cost of RAM.
    On Colab with 12 GB RAM: cache_rate=0.5 is safe. On TPU/A100: 1.0.

    Returns (train_loader, val_loader)
    """
    all_data = build_brats_data_list(root, max_cases=max_cases)

    # 90/10 split
    split_idx  = int(len(all_data) * (1 - val_fraction))
    train_data = all_data[:split_idx]
    val_data   = all_data[split_idx:]

    train_ds = CacheDataset(
        data      = train_data,
        transform = get_brats_transforms(
            target_spacing = CFG["VOXEL_SPACING"],
            slice_size     = CFG["SLICE_SIZE"],
            is_train       = True,
        ),
        cache_rate    = CFG["CACHE_RATE"],
        num_workers   = CFG["NUM_WORKERS"],
        progress      = True,
    )

    val_ds = CacheDataset(
        data      = val_data,
        transform = get_brats_transforms(
            target_spacing = CFG["VOXEL_SPACING"],
            slice_size     = CFG["SLICE_SIZE"],
            is_train       = False,
        ),
        cache_rate    = CFG["CACHE_RATE"],
        num_workers   = CFG["NUM_WORKERS"],
        progress      = True,
    )

    train_loader = DataLoader(
        train_ds,
        batch_size  = CFG["BATCH_SIZE"],
        shuffle     = True,
        num_workers = CFG["NUM_WORKERS"],
        pin_memory  = True,
        drop_last   = True,
    )

    val_loader = DataLoader(
        val_ds,
        batch_size  = CFG["BATCH_SIZE"],
        shuffle     = False,
        num_workers = CFG["NUM_WORKERS"],
        pin_memory  = True,
    )

    print(f"[BraTS] Train: {len(train_ds)} | Val: {len(val_ds)}")
    return train_loader, val_loader


def build_remind_dataloaders(
    root: str,
    val_fraction: float = 0.1,
) -> Tuple[DataLoader, DataLoader]:
    """
    Build CachedDataset-backed DataLoaders for ReMIND2Reg.

    Returns (train_loader, val_loader)
    """
    all_data  = build_remind_data_list(root)
    split_idx = int(len(all_data) * (1 - val_fraction))
    train_data = all_data[:split_idx]
    val_data   = all_data[split_idx:]

    train_ds = CacheDataset(
        data      = train_data,
        transform = get_remind_transforms(
            target_spacing = CFG["VOXEL_SPACING"],
            slice_size     = CFG["SLICE_SIZE"],
            is_train       = True,
        ),
        cache_rate  = CFG["CACHE_RATE"],
        num_workers = CFG["NUM_WORKERS"],
        progress    = True,
    )

    val_ds = CacheDataset(
        data      = val_data,
        transform = get_remind_transforms(
            target_spacing = CFG["VOXEL_SPACING"],
            slice_size     = CFG["SLICE_SIZE"],
            is_train       = False,
        ),
        cache_rate  = CFG["CACHE_RATE"],
        num_workers = CFG["NUM_WORKERS"],
        progress    = True,
    )

    train_loader = DataLoader(
        train_ds,
        batch_size  = max(1, CFG["BATCH_SIZE"] // 2),  # Smaller batch: bigger volumes
        shuffle     = True,
        num_workers = CFG["NUM_WORKERS"],
        pin_memory  = True,
        drop_last   = True,
    )

    val_loader = DataLoader(
        val_ds,
        batch_size  = max(1, CFG["BATCH_SIZE"] // 2),
        shuffle     = False,
        num_workers = CFG["NUM_WORKERS"],
        pin_memory  = True,
    )

    print(f"[ReMIND2Reg] Train: {len(train_ds)} | Val: {len(val_ds)}")
    return train_loader, val_loader


## 1.7 — QUICK SANITY CHECK (Run standalone to verify pipeline)

In [9]:
def verify_pipeline() -> None:
    """
    Load one sample from ReMIND2Reg and print tensor statistics.
    Run this cell in isolation to verify the pipeline before training.
    """
    print("=" * 60)
    print("PHASE 1: Pipeline Sanity Check")
    print("=" * 60)

    # Test ReMIND2Reg pipeline with a single sample
    single_sample = build_remind_data_list(CFG["ROOT_REMIND"])[:1]
    if not single_sample:
        print("[WARN] No valid ReMIND2Reg samples found. Check root path.")
        return

    test_ds = CacheDataset(
        data      = single_sample,
        transform = get_remind_transforms(is_train=False, slice_size=256),
        cache_rate = 1.0,
        num_workers = 0,
    )
    sample = test_ds[0]

    img = sample["image"]   # (3, 256, 256) tensor
    us  = sample["us"]      # (3, 256, 256) tensor

    print(f"  [MR  ] shape={img.shape}  dtype={img.dtype}  "
          f"min={img.min():.3f}  max={img.max():.3f}  mean={img.mean():.3f}")
    print(f"  [iUS ] shape={us.shape}   dtype={us.dtype}   "
          f"min={us.min():.3f}   max={us.max():.3f}   mean={us.mean():.3f}")
    print(f"\n  Patient ID: {sample.get('patient_id', 'N/A')}")
    print(f"  Estimated GPU memory per batch of 8: "
          f"{8 * 3 * 256 * 256 * 2 / 1e6:.1f} MB (FP16)")
    print("\n  [OK] Phase 1 pipeline verified.")


if __name__ == "__main__":
    verify_pipeline()

PHASE 1: Pipeline Sanity Check
[ReMIND2Reg] Found 0 usable patient cases.
[WARN] No valid ReMIND2Reg samples found. Check root path.
